In [1]:
from model.model import get_model
import os
from dotenv import load_dotenv
from schema.ticket import Ticket 
from prompt.summarizer import summary_prompt
import outlines
from schema.table import Table
import json
from evalution.metric import TicketEvaluator
load_dotenv()

True

In [2]:
location = os.getenv('DATA_FILE_NAME')
table_1 = Table(location)

In [3]:
name = os.getenv('MODEL_NAME')
hf_model,hf_tokenizer = get_model(name)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

model downloaded


In [4]:
hf_model.eval()

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [5]:
model = outlines.from_transformers(hf_model,hf_tokenizer)

In [6]:
#generator = model("My laptop screen is broken and I need urgent help.",output_type=Ticket,max_new_tokens=200,temperature=0.2,do_sample=True)

In [7]:
#generator

In [8]:
evaluator = TicketEvaluator(generator=model)


In [9]:
#metric = evaluator.evalute_ticket(table_1.return_ticket(1),max_new_tokens=150,use_cache= True)

In [19]:
df = evaluator.evalutaion_dataframe(output_type=Ticket,max_new_tokens=200,temperature=0.3,top_k=2)

ticket_id                                                    1
message      My laptop screen is broken and I need urgent h...
category                                             technical
sentiment                                             negative
urgency                                                   high
Name: 0, dtype: object
ticket_id                                             2
message      I forgot my password and need to reset it.
category                                        account
sentiment                                       neutral
urgency                                          medium
Name: 1, dtype: object


KeyboardInterrupt: 

In [11]:
df

,ticket_id,message,category,sentiment,urgency,success,pred_category,pred_sentiment,pred_urgency,pred_summary
0,1,My laptop screen is broken and I need urgent h...,technical,negative,high,True,technical,positive,high,Customer needs immediate assistance with their...
1,2,I forgot my password and need to reset it.,account,neutral,medium,True,account,neutral,low,Customer needs assistance with resetting their...
2,3,My package arrived two days late.,delivery,negative,medium,True,delivery,negative,high,Package delivery was delayed due to unforeseen...
3,4,I was charged twice for the same order.,billing,negative,high,True,technical,negative,high,Order charge discrepancy
4,5,I want to cancel my subscription.,subscription,neutral,medium,True,subscription,neutral,low,Customer wants to terminate their current subs...
5,6,"Your support team was very helpful, thank you!",account,positive,low,True,subscription,positive,low,The customer expressed gratitude for their sup...
6,7,The payment failed but money was deducted from...,billing,negative,high,True,technical,negative,low,"Payment failed, but funds were successfully de..."
7,8,I received my order early and everything looks...,delivery,positive,low,True,subscription,neutral,low,The customer is satisfied with their subscript...
8,9,The mobile app freezes when I upload a file.,technical,negative,high,True,technical,negative,high,The user is experiencing technical issues with...
9,10,How can I upgrade my subscription plan?,subscription,neutral,low,True,subscription,positive,low,The customer is seeking information on upgradi...


In [12]:
metric=evaluator.accuracy_metric(df)

clean_metric = {k:float(v) for k,v in metric.items()}

In [13]:
clean_metric

{'category_accuracy': 60.0,
 'sentiment_accuracy': 70.0,
 'urgency_accuracy': 60.0}

In [14]:
#result = generator(summary_prompt(table_1.return_ticket(2)), max_new_tokens=150,use_cache= True)
#print(result)

In [15]:
#output = Ticket.model_validate_json(result)
#output = output.model_dump_json()

In [16]:
#output = json.loads(output)

In [17]:
#output['category']